# Company AI Agent

Local Autonomous AI Agent for WhatsApp

- Arabic and English support
- Qwen3 via Ollama
- Excel / Google Sheets
- Customer memory
- RAG search
- Tool calling
- Controlled business operations
- WhatsApp Cloud API

In [ ]:
%pip install -q \
    requests \
    python-dotenv \
    flask \
    gspread \
    google-auth \
    openpyxl \
    rank-bm25 \
    sentence-transformers \
    apscheduler \
    nest-asyncio

In [ ]:
import os
import re
import json
import time
import hmac
import hashlib
import sqlite3
import logging
import threading
import requests

from datetime import datetime, timezone
from pathlib import Path
from dotenv import load_dotenv

from flask import Flask, request

from openpyxl import load_workbook
from rank_bm25 import BM25Okapi

from google.oauth2.service_account import Credentials
import gspread

from concurrent.futures import ThreadPoolExecutor

load_dotenv()

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(message)s"
)

logger = logging.getLogger("CompanyAgent")

print("Libraries loaded successfully")

In [ ]:
CONFIG = {
    # Local AI
    "OLLAMA_URL": "http://127.0.0.1:11434",
    "OLLAMA_MODEL": "qwen3:8b",

    # Data source: excel or google_sheets
    "SOURCE_TYPE": "excel",

    "EXCEL_FILE": "company.xlsx",
    "EXCEL_SHEET": None,

    "GOOGLE_SHEET_ID": "",
    "GOOGLE_CREDENTIALS": "service_account.json",

    # WhatsApp
    "WHATSAPP_TOKEN": "",
    "PHONE_NUMBER_ID": "",
    "WHATSAPP_VERIFY_TOKEN": "",
    "WHATSAPP_APP_SECRET": "",
    "WA_API_VERSION": "vXX.X",

    # Database
    "DATABASE": "company_agent.db",

    # Agent
    "MAX_TOOL_STEPS": 5,
    "MAX_HISTORY": 12,

    # Safety
    "MAX_DAILY_MESSAGES_PER_CUSTOMER": 100,
    "MAX_PARALLEL_TASKS": 4,

    # Search
    "SEARCH_RESULTS": 6
}

print("Configuration loaded")

In [ ]:
def check_ollama():

    url = CONFIG["OLLAMA_URL"] + "/api/tags"

    response = requests.get(url, timeout=10)
    response.raise_for_status()

    models = response.json().get("models", [])

    print("Available models:")

    for model in models:
        print("-", model["name"])

    expected = CONFIG["OLLAMA_MODEL"]

    if not any(
        m["name"] == expected
        for m in models
    ):
        print(f"\nModel {expected} is not installed.")
        print(f"Run: ollama pull {expected}")
    else:
        print("\nModel is ready!")

check_ollama()